In [ ]:
#this cell ensures that the updated version of the collegebaseball library is loaded.
from pathlib import Path
import sys
import pandas as pd

# --- Assume notebook is run from:
# <repo>/SCRAPERS/Game_Stat_Scraper/notebooks/   OR
# <repo>/SCRAPERS/Game_Stat_Scraper/visualizations/
#
# In both cases, Game_Stat_Scraper root is the parent of CWD.
GAME_SCRAPER_ROOT = Path.cwd().resolve().parent
VENDORED_CB_ROOT = GAME_SCRAPER_ROOT / "vendor" / "collegebaseball"   # this folder contains the "collegebaseball/" package

assert VENDORED_CB_ROOT.exists(), f"Vendored collegebaseball not found at: {VENDORED_CB_ROOT}"

# Put vendored package FIRST on sys.path (ahead of site-packages)
if str(VENDORED_CB_ROOT) not in sys.path:
    sys.path.insert(0, str(VENDORED_CB_ROOT))

# Now import and verify exactly where it came from
import collegebaseball
from collegebaseball import lookup, ncaa_scraper, guts

print("collegebaseball imported from:", Path(collegebaseball.__file__).resolve())

# Verify which seasons.csv you are using
seasons_path = Path(collegebaseball.__file__).resolve().parent / "data" / "seasons.csv"
print("seasons.csv path:", seasons_path)
display(pd.read_csv(seasons_path).tail(10))

In [ ]:
import os
import requests
from bs4 import BeautifulSoup
import re
from playwright.async_api import async_playwright
import asyncio
from urllib.parse import urlencode
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import inspect
from collections import defaultdict
from functools import reduce


In [ ]:
# Portable paths (assumes notebook is run from scrapers/game_stat_scraper/notebooks/)
NB_DIR = Path.cwd()
GAME_SCRAPER_ROOT = NB_DIR.parent                  # .../scrapers/game_stat_scraper
PROJECT_ROOT = GAME_SCRAPER_ROOT.parent.parent     # repo root

# Vendored collegebaseball fork inside the repo
REPO_ROOT = GAME_SCRAPER_ROOT / "vendor" / "collegebaseball"
if not REPO_ROOT.exists():
    raise FileNotFoundError(f"Expected vendored collegebaseball at: {REPO_ROOT}")

# Make vendored package importable
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("REPO_ROOT:", REPO_ROOT)
print("Python:", sys.executable)


In [ ]:
print("Python:", sys.executable)
print("collegebaseball:", collegebaseball.__file__)

## setting up scraper to obtain the season IDs required to extend the range of this scraping library further than 2013-2023

### this started with finding a page that can scrape season IDs that match the ones already in use with the library's hardcoded seasons csv

### then once I found the matching IDs,  I extended it to the URLs for more recent seasons to then extract those IDs and update the csv that the library utilizes

### I also had to modify the data loading functions the library uses to prevent it from returning certain team-level advanced stats that rely on weights stored in a hard-coded CSV file. This way, the functions will only return data directly on the website, not data derived from weights, which we only have for a limited set of seasons. 

In [ ]:
# 1) Load seasons.parquet the same way the package does
seasons_df = guts.get_seasons_table()
seasons_df

In [ ]:
async def extract_game_sport_year_ctl_id(team_year_url: str):
    """
    Opens a /teams/<team_year_id>/... page, clicks Team Statistics if needed,
    then finds any /team/<x>/stats/<y> URL and returns y as game_sport_year_ctl_id.
    """
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False, slow_mo=50)
        ctx = await browser.new_context()
        page = await ctx.new_page()

        await page.goto(team_year_url, wait_until="domcontentloaded")
        await page.wait_for_timeout(1200)

        # If "Team Statistics" exists, click it to load the stats subpage
        ts = page.get_by_role("link", name=re.compile(r"Team Statistics", re.I))
        if await ts.count() > 0:
            await ts.first.click()
            await page.wait_for_load_state("domcontentloaded")
            await page.wait_for_timeout(1200)

        # 1) First: see if the current URL itself is a /team/.../stats/... page
        m = re.search(r"/team/\d+/stats/(\d+)", page.url)
        if m:
            season_id = int(m.group(1))
            await browser.close()
            return season_id, page.url

        # 2) Otherwise: scan all anchors for /team/<x>/stats/<y>
        anchors = page.locator("a[href]")
        n = await anchors.count()

        matches = []
        for i in range(n):
            href = await anchors.nth(i).get_attribute("href")
            if not href:
                continue
            mm = re.search(r"/team/\d+/stats/(\d+)", href)
            if mm:
                matches.append((int(mm.group(1)), href))

        if not matches:
            # 3) Last resort: the link might be created by JS and not in hrefs.
            # Dump any URLs in the DOM text that match the pattern.
            html = await page.content()
            mm = re.search(r"/team/\d+/stats/(\d+)", html)
            if mm:
                season_id = int(mm.group(1))
                await browser.close()
                return season_id, f"(found-in-html) {mm.group(0)}"

            await browser.close()
            raise RuntimeError("Could not find any /team/<x>/stats/<y> reference on the page.")

        # If multiple, pick the first (or you can print them all)
        season_id, href = matches[0]
        await browser.close()
        return season_id, href


In [ ]:
season_id, where = await extract_game_sport_year_ctl_id("https://stats.ncaa.org/teams/548218/season_to_date_stats")
print(season_id, where)

### found the matching season id for exisitng supported seasons in library using above approach, so below is extending it to retrieve these id's for more recent seasons not initially supported in the library

In [ ]:
PLAYER_GBG_URL = "https://stats.ncaa.org/players/7836437"

async def extract_ids_from_players_page(url=PLAYER_GBG_URL, headed=True):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=not headed, slow_mo=50 if headed else 0)
        ctx = await browser.new_context()
        page = await ctx.new_page()

        await page.goto(url, wait_until="domcontentloaded")

        # Wait for the season dropdown (this is the key)
        # NCAA pages sometimes render it slightly differently, so use a loose selector
        await page.wait_for_selector("select", timeout=15000)

        html = await page.content()
        await browser.close()

    soup = BeautifulSoup(html, "lxml")
    
    print("All <select> elements found on page:")
    for s in soup.find_all("select"):
        print("name:", s.get("name"), "| id:", s.get("id"), "| option_count:", len(s.find_all("option")))

    # Find a select that looks like the season dropdown
    # Prefer exact NCAA param name if present
    season_sel = (
        soup.select_one("select[name='game_sport_year_ctl_id']")
        or soup.select_one("select#game_sport_year_ctl_id")
    )

    # Find category select (sometimes exists, sometimes it’s encoded as links/buttons)
    cat_sel = (
        soup.select_one("select[name='year_stat_category_id']")
        or soup.select_one("select#year_stat_category_id")
    )

    seasons = []
    if season_sel:
        for o in season_sel.select("option[value]"):
            seasons.append((o.get_text(strip=True), o["value"].strip()))

    cats = []
    if cat_sel:
        for o in cat_sel.select("option[value]"):
            cats.append((o.get_text(strip=True), o["value"].strip()))

    # Backup: sometimes categories are in hrefs or data-* attributes (tabs/dropdowns)
    # Pull any year_stat_category_id occurrences from the HTML
    cat_ids_in_html = sorted(set(re.findall(r"year_stat_category_id[=:/](\d+)", html)))

    return {
        "found_season_select": season_sel is not None,
        "found_category_select": cat_sel is not None,
        "seasons": seasons,
        "categories": cats,
        "category_ids_anywhere_in_html": cat_ids_in_html,
    }



In [ ]:
out = await extract_ids_from_players_page()
out

In [ ]:
TEAM_HUB = "https://stats.ncaa.org/team/457"
ORG_ID = 457

async def scrape_all_season_ids(start_year=2025, end_year=2020):
    results = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False, slow_mo=40)
        context = await browser.new_context()
        page = await context.new_page()

        # STEP 1 — Go to team hub
        await page.goto(TEAM_HUB, wait_until="domcontentloaded")

        # STEP 2 — Click current year Baseball link (default season)
        await page.locator("text=Baseball").first.click()
        await page.wait_for_load_state("domcontentloaded")

        # We are now on Schedule/Results page for latest season

        # Grab dropdown seasons list
        await page.wait_for_selector("select#year_list")
        season_dropdown = page.locator("select#year_list")

        options = await season_dropdown.locator("option").all()
        season_map = {}

        for opt in options:
            label = (await opt.inner_text()).strip()
            value = await opt.get_attribute("value")
            season_map[label] = value

        # Filter seasons in desired range
        target_seasons = []
        for label in season_map:
            match = re.search(r"(\d{4})", label)
            if match:
                year = int(match.group(1))
                if end_year <= year <= start_year:
                    target_seasons.append(label)

        # Sort descending newest → oldest
        target_seasons = sorted(
            target_seasons,
            key=lambda x: int(re.search(r"(\d{4})", x).group(1)),
            reverse=True
        )

        for season_label in target_seasons:
            print(f"\nProcessing {season_label}")

            # STEP 7 — Select season from dropdown
            await season_dropdown.select_option(season_map[season_label])
            await page.wait_for_load_state("domcontentloaded")

            # We are back on Schedule/Results tab

            # STEP 3 — Click "Team Statistics"
            await page.locator("text=Team Statistics").click()
            await page.wait_for_load_state("domcontentloaded")

            # STEP 4 — Extract season ID from HTML
            html = await page.content()
            soup = BeautifulSoup(html, "lxml")

            season_id_match = re.search(
                rf"/team/{ORG_ID}/stats/(\d+)", html
            )

            if not season_id_match:
                print("Could not find season_id")
                continue

            season_id = season_id_match.group(1)
            print("  season_id:", season_id)

            # STEP 5 — Click "Game By Game"
            await page.locator("text=Game By Game").click()
            await page.wait_for_load_state("domcontentloaded")

            # STEP 6 — Extract category IDs
            gbg_html = await page.content()

            category_ids = sorted(set(
                re.findall(r"year_stat_category_id[=:/](\d+)", gbg_html)
            ))

            print("  category_ids:", category_ids)

            results.append({
                "season_label": season_label,
                "season_id": season_id,
                "category_ids": category_ids
            })

        await browser.close()

    return results

In [ ]:
results = await scrape_all_season_ids()
results

### above is the result of scraping the season id's for all more recent seasons that weren't initially supported (beyond 2023)

In [ ]:
def label_to_season_int(season_label: str) -> int:
    # "2025-26" -> 2026
    start, end2 = season_label.split("-")
    return int(start[:2] + end2)

In [ ]:
mapped_rows = []

for rec in results:
    season_int = label_to_season_int(rec["season_label"])
    batting_id, pitching_id, fielding_id = map(int, rec["category_ids"])
    
    mapped_rows.append({
        "season": season_int,
        "season_id": int(rec["season_id"]),
        "batting_id": batting_id,
        "pitching_id": pitching_id,
        "fielding_id": fielding_id
    })

mapped_rows

In [ ]:
df_new = pd.DataFrame(mapped_rows).sort_values("season")
df_new

In [ ]:
REPO_ROOT = REPO_ROOT  # set in PATHS cell above

# Paths in this repo (based on the screenshot / typical layout)
CSV_PATH = REPO_ROOT / "collegebaseball" / "data" / "seasons.csv"
PARQUET_PATH = REPO_ROOT / "collegebaseball" / "data" / "seasons.parquet"

# 1) take only 2024-2026 from df_new
patch = (
    df_new[df_new["season"].isin([2024, 2025, 2026])]
    .copy()
    .astype({"season": "int32", "season_id": "int32", "batting_id": "int32", "pitching_id": "int32", "fielding_id": "int32"})
)

patch

### rewriting the seasons csv and seasons parquet that the library utilizes to support more recent seasons

In [ ]:
# Read existing
seasons_csv = pd.read_csv(CSV_PATH)

# Ensure consistent dtypes / columns
keep_cols = ["season", "season_id", "batting_id", "pitching_id", "fielding_id"]
seasons_csv = seasons_csv[keep_cols].copy()

# Upsert by season (patch overwrites)
updated_csv = (
    pd.concat([seasons_csv, patch], ignore_index=True)
      .drop_duplicates(subset=["season"], keep="last")
      .sort_values("season")
      .reset_index(drop=True)
)

# Write back
updated_csv.to_csv(CSV_PATH, index=False)

print("Wrote:", CSV_PATH)
updated_csv.tail(10)

In [ ]:
# Read existing
seasons_pq = pd.read_parquet(PARQUET_PATH)

# Keep same schema
seasons_pq = seasons_pq[keep_cols].copy()

# Upsert
updated_pq = (
    pd.concat([seasons_pq, patch], ignore_index=True)
      .drop_duplicates(subset=["season"], keep="last")
      .sort_values("season")
      .reset_index(drop=True)
)

# Write back
updated_pq.to_parquet(PARQUET_PATH, index=False)

print("Wrote:", PARQUET_PATH)
updated_pq.tail(10)

### verifying that i have updated the seasons csv's with the correct season id's for more recent seasons

In [ ]:
# Re-load and verify the 2024-2026 rows are correct
check_csv = pd.read_csv(CSV_PATH)
check_pq  = pd.read_parquet(PARQUET_PATH)

print("CSV rows 2024+:")
display(check_csv[check_csv["season"] >= 2024])

print("PARQUET rows 2024+:")
display(check_pq[check_pq["season"] >= 2024])

In [ ]:
lookup._lookup_season_info(2026)

## now that i have retrieved the season id's to extend this scraping library, pivoting the loading the data and figuring out which variables are consistenlty available throughout all seasons and which need to be dropped to maintain full coverage of all variables

### the first step of this is implementing the headless = false patch to work around the way that stats.ncaa.org blocks headless automated browser activity

In [ ]:
def _make_driver_headed():
    opts = webdriver.ChromeOptions()
    opts.add_argument("--window-size=1920,1080")
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=opts)

_driver = _make_driver_headed()

_OrigSession = ncaa_scraper.Session
_orig_get = _OrigSession.get

class _FakeResp:
    def __init__(self, text, status_code=200, url=""):
        self.text = text
        self.status_code = status_code
        self.url = url

def _selenium_get(self, url, params=None, headers=None, **kwargs):
    full_url = url
    if params:
        full_url = f"{url}{'&' if '?' in url else '?'}{urlencode(params)}"

    _driver.get(full_url)
    html = _driver.page_source

    if "Access Denied" in html or "You don't have permission" in html:
        print("Blocked URL:", full_url)
        return _FakeResp(html, status_code=403, url=full_url)

    return _FakeResp(html, status_code=200, url=full_url)

# apply patch
_OrigSession.get = _selenium_get
print("Patched collegebaseball Session.get -> Selenium headed browser.")

def unpatch_and_quit():
    _OrigSession.get = _orig_get
    try:
        _driver.quit()
    except Exception:
        pass
    print("Unpatched Session.get and quit Selenium driver.")

In [ ]:
print(collegebaseball.__file__)
print(lookup.__file__)
print(ncaa_scraper.__file__)

### loading each data end point we have for each team/season below

In [ ]:
school = "North Carolina"

df_res_2026 = ncaa_scraper.ncaa_team_results(school, 2026)
print(df_res_2026.shape)
display(df_res_2026.head())

In [ ]:
df_res_2026

In [ ]:
variants = ["batting", "pitching", "fielding"]
for v in variants:
    df = ncaa_scraper.ncaa_team_game_logs(
        school=school,
        season=2026,
        variant=v,
        include_advanced=False,   # turning off advanced metrics that rely on annual hardcoded weights each season
    )
    #print(v, df.shape)
    #display(df.head())
    print(df.columns)
    df

In [ ]:
school = "North Carolina"
df = ncaa_scraper.ncaa_team_game_logs(
        school=school,
        season=2026,
        variant= "batting",
        include_advanced=False,
)


In [ ]:
df

In [ ]:
df[['date', 'opponent_name', 'game_id']]

In [ ]:
# Checking number of rows and columns for each gamelog of each season for a team
seasons = list(range(2013, 2027))
variants = ["batting", "pitching", "fielding"]

results_by_year = {}
gamelogs_by_variant_year = defaultdict(dict)

failures = []

for yr in seasons:
    try:
        df = ncaa_scraper.ncaa_team_results(school, yr)
        results_by_year[yr] = df
        print(f"results {yr}: {df.shape}")
    except Exception as e:
        failures.append(("results", yr, repr(e)))
        print(f"results {yr}: FAILED -> {e}")

for v in variants:
    for yr in seasons:
        try:
            df = ncaa_scraper.ncaa_team_game_logs(
                school=school,
                season=yr,
                variant=v,
                include_advanced=False,
            )
            gamelogs_by_variant_year[v][yr] = df
            print(f"gamelogs {v} {yr}: {df.shape}")
        except Exception as e:
            failures.append((v, yr, repr(e)))
            print(f"gamelogs {v} {yr}: FAILED -> {e}")

failures[:10], len(failures)

In [ ]:
# completeness checks
missing_results_years = [y for y in seasons if y not in results_by_year]
if missing_results_years:
    raise RuntimeError(f"Missing results years: {missing_results_years}")

for v in variants:
    missing = [y for y in seasons if y not in gamelogs_by_variant_year[v]]
    if missing:
        raise RuntimeError(f"Missing gamelog years for {v}: {missing}")

# intersections across ALL seasons
common_results_cols = set.intersection(*[set(results_by_year[y].columns) for y in seasons])

common_gamelog_cols = {
    v: set.intersection(*[set(gamelogs_by_variant_year[v][y].columns) for y in seasons])
    for v in variants
}

common_all_variants_cols = set.intersection(
    common_gamelog_cols["batting"],
    common_gamelog_cols["pitching"],
    common_gamelog_cols["fielding"],
)

print("Common results cols:", len(common_results_cols))
for v in variants:
    print(f"Common {v} gamelog cols:", len(common_gamelog_cols[v]))
print("Common across all 3 variants:", len(common_all_variants_cols))

# Filtered dataframes
common_results_cols_sorted = [c for c in results_by_year[seasons[0]].columns if c in common_results_cols]
results_filtered = {y: results_by_year[y][common_results_cols_sorted].copy() for y in seasons}

gamelogs_filtered = {v: {} for v in variants}
for v in variants:
    cols_sorted = [c for c in gamelogs_by_variant_year[v][seasons[0]].columns if c in common_gamelog_cols[v]]
    for y in seasons:
        gamelogs_filtered[v][y] = gamelogs_by_variant_year[v][y][cols_sorted].copy()

In [ ]:
inconsistent_report = {}

for v in variants:
    season_cols = {
        y: set(gamelogs_by_variant_year[v][y].columns)
        for y in seasons
    }

    union_cols = reduce(set.union, season_cols.values())
    intersection_cols = reduce(set.intersection, season_cols.values())

    inconsistent = union_cols - intersection_cols

    inconsistent_report[v] = {
        "union": union_cols,
        "intersection": intersection_cols,
        "inconsistent": inconsistent,
        "season_cols": season_cols
    }

    print(f"\n=== {v.upper()} ===")
    print("Total columns ever seen:", len(union_cols))
    print("Columns present in ALL seasons:", len(intersection_cols))
    print("Columns NOT present in every season:", len(inconsistent))

In [ ]:
for v in variants:
    print(f"\n===== {v.upper()} INCONSISTENCIES =====")

    for col in sorted(inconsistent_report[v]["inconsistent"]):
        missing_years = [
            y for y in seasons
            if col not in inconsistent_report[v]["season_cols"][y]
        ]
        print(f"{col} → missing in seasons: {missing_years}")

In [ ]:
# Build result column sets
results_cols = {
    y: set(results_by_year[y].columns)
    for y in seasons
}

results_union = reduce(set.union, results_cols.values())
results_intersection = reduce(set.intersection, results_cols.values())

print("=== RESULTS VARIABLES ===")
print("\nTotal unique columns ever seen:", len(results_union))
print(sorted(results_union))

print("\nColumns present in ALL seasons:", len(results_intersection))
print(sorted(results_intersection))

In [ ]:
for v in variants:
    print(f"\n=== {v.upper()} COMMON VARIABLES (ALL SEASONS) ===")
    common_cols = sorted(inconsistent_report[v]["intersection"])
    print(f"Count: {len(common_cols)}")
    for col in common_cols:
        print(col)

In [ ]:
# Results columns (should be identical across years)
print("RESULTS columns:")
print(list(results_by_year[seasons[0]].columns))

# Gamelog columns by variant and year
for v in variants:
    print(f"\n=== {v.upper()} columns (example year {seasons[0]}) ===")
    print(list(gamelogs_by_variant_year[v][seasons[0]].columns))

### starting code to clean up the data returned from stats.ncaa.org for completeness from 2013-current time

In [ ]:
# Batting variant clean up
def standardize_batting(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # 1) Canonicalize DP naming
    # Older seasons use "DP"; newer seasons use "OPP DP"
    if "OPP DP" in df.columns and "DP" not in df.columns:
        df = df.rename(columns={"OPP DP": "opp_dp"})
    elif "DP" in df.columns and "OPP DP" not in df.columns:
        df = df.rename(columns={"DP": "opp_dp"})
    elif "DP" in df.columns and "OPP DP" in df.columns:
        # Rare, but if both ever appear, pick one canonical and drop the other
        df = df.rename(columns={"OPP DP": "opp_dp"})
        df = df.drop(columns=["DP"])

    # 2) Drop GDP if present (only tracked in more recent years)
    if "GDP" in df.columns:
        df = df.drop(columns=["GDP"])

    return df

In [ ]:
# Pitching variant clean up
def standardize_pitching(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    drop_cols = [c for c in ["CG", "pickoffs"] if c in df.columns]
    if drop_cols:
        df = df.drop(columns=drop_cols)
    return df

In [ ]:
# Fielding variant clean up
def standardize_fielding(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Make sure PO/A/E are numeric
    for c in ["PO", "A", "E"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    # Compute TC robustly: treat missing PO/A/E as 0 for the sum,
    # but if all three are NaN, keep TC as NaN.
    po = df["PO"] if "PO" in df.columns else pd.Series([pd.NA]*len(df))
    a  = df["A"]  if "A"  in df.columns else pd.Series([pd.NA]*len(df))
    e  = df["E"]  if "E"  in df.columns else pd.Series([pd.NA]*len(df))

    all_nan = po.isna() & a.isna() & e.isna()
    tc = po.fillna(0) + a.fillna(0) + e.fillna(0)
    tc = tc.astype("float")
    tc[all_nan] = pd.NA

    df["TC"] = tc
    return df

In [ ]:
# apply standardization to all seasons
standardized = {v: {} for v in variants}

for y in seasons:
    standardized["batting"][y]  = standardize_batting(gamelogs_by_variant_year["batting"][y])
    standardized["pitching"][y] = standardize_pitching(gamelogs_by_variant_year["pitching"][y])
    standardized["fielding"][y] = standardize_fielding(gamelogs_by_variant_year["fielding"][y])

In [ ]:
# rechecking consistency
for v in variants:
    cols_by_year = {y: set(standardized[v][y].columns) for y in seasons}
    union_cols = reduce(set.union, cols_by_year.values())
    inter_cols = reduce(set.intersection, cols_by_year.values())
    print(f"{v}: union={len(union_cols)} intersection={len(inter_cols)} inconsistent={len(union_cols - inter_cols)}")

In [ ]:
unc_school_id = 457

In [ ]:
unc_res_2013 = ncaa_scraper.ncaa_team_results(school=unc_school_id, season=2013)
unc_res_2013.head()

In [ ]:
unc_res_2013[unc_res_2013["date"].isin(["03/23/2013","04/01/2013","06/01/2013","06/02/2013","06/09/2013"])]